Caso Univariado

In [1]:
import numpy as np
import math

# separa dados por classe
def separar_por_classe(X, y):
    classes = {}
    for i in range(len(y)):
        classe = y[i]
        if classe not in classes:
            classes[classe] = []
        classes[classe].append(X[i])
    return classes

# calcula média e desvio padrão por atributo
def resumo_dataset(dataset):
    resumo = []
    dataset = np.array(dataset)
    
    for i in range(dataset.shape[1]):
        col = dataset[:, i]
        media = np.mean(col)
        desvio = np.std(col)
        resumo.append((media, desvio))
    
    return resumo

# treinar
def treinar_naive_bayes_univariado(X, y):
    separados = separar_por_classe(X, y)
    modelo = {}
    
    for classe, dados in separados.items():
        modelo[classe] = resumo_dataset(dados)
    
    return modelo

# pdf gaussiana
def probabilidade_gaussiana(x, media, desvio):
    if desvio == 0:
        return 1 if x == media else 1e-9
    
    expoente = math.exp(-((x - media) ** 2) / (2 * desvio ** 2))
    return (1 / (math.sqrt(2 * math.pi) * desvio)) * expoente

# prever uma amostra
def prever_univariado(modelo, x):
    probabilidades = {}
    
    for classe, resumo in modelo.items():
        prob = 1
        for i in range(len(resumo)):
            media, desvio = resumo[i]
            prob *= probabilidade_gaussiana(x[i], media, desvio)
        
        probabilidades[classe] = prob
    
    return max(probabilidades, key=probabilidades.get)

Caso Multivariado

In [ ]:
# treinar
def treinar_bayes_multivariado(X, y):
    separados = separar_por_classe(X, y)
    modelo = {}
    
    for classe, dados in separados.items():
        dados = np.array(dados)
        
        mu = np.mean(dados, axis=0)
        sigma = np.cov(dados, rowvar=False)
        
        modelo[classe] = {
            "mu": mu,
            "sigma": sigma,
            "inv": np.linalg.inv(sigma),
            "det": np.linalg.det(sigma)
        }
    
    return modelo

# pdf multivariada
def prob_multivariada(x, mu, sigma_inv, sigma_det):
    d = len(x)
    x_mu = x - mu
    
    expoente = -0.5 * np.dot(np.dot(x_mu.T, sigma_inv), x_mu)
    
    coef = 1 / (((2 * np.pi) ** (d / 2)) * np.sqrt(sigma_det))
    
    return coef * np.exp(expoente)

# prever
def prever_multivariado(modelo, x):
    probabilidades = {}
    
    for classe, params in modelo.items():
        prob = prob_multivariada(
            x,
            params["mu"],
            params["inv"],
            params["det"]
        )
        probabilidades[classe] = prob
    
    return max(probabilidades, key=probabilidades.get)

In [28]:
def treinar_bayes_multivariado(X, y):
    separados = separar_por_classe(X, y)
    modelo = {}
    
    for classe, dados in separados.items():
        dados = np.array(dados)
        
        mu = np.mean(dados, axis=0)
        sigma = np.cov(dados, rowvar=False)
        
        # 🔥 correção aqui
        sigma += np.eye(sigma.shape[0]) * 1e-6
        
        modelo[classe] = {
            "mu": mu,
            "sigma": sigma,
            "inv": np.linalg.inv(sigma),
            "det": np.linalg.det(sigma)
        }
    
    return modelo

aplicacao

In [3]:
def accuracy(y_true, y_pred):
    corretos = 0
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            corretos += 1
    return corretos / len(y_true)

In [4]:
def precision_recall_f1(y_true, y_pred):
    classes = list(set(y_true))
    
    precisions = []
    recalls = []
    f1s = []
    
    for classe in classes:
        tp = fp = fn = 0
        
        for i in range(len(y_true)):
            if y_pred[i] == classe and y_true[i] == classe:
                tp += 1
            elif y_pred[i] == classe and y_true[i] != classe:
                fp += 1
            elif y_pred[i] != classe and y_true[i] == classe:
                fn += 1
        
        precision = tp / (tp + fp) if (tp + fp) != 0 else 0
        recall = tp / (tp + fn) if (tp + fn) != 0 else 0
        
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) != 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    
    # média macro
    return (
        sum(precisions) / len(precisions),
        sum(recalls) / len(recalls),
        sum(f1s) / len(f1s)
    )

avaliacao univariado

In [5]:
def avaliar_univariado(X_train, y_train, X_test, y_test):
    modelo = treinar_naive_bayes_univariado(X_train, y_train)
    
    y_pred = []
    for x in X_test:
        y_pred.append(prever_univariado(modelo, x))
    
    acc = accuracy(y_test, y_pred)
    prec, rec, f1 = precision_recall_f1(y_test, y_pred)
    
    return acc, prec, f1

In [30]:
import time
def avaliar_univariado(X_train, y_train, X_test, y_test):
    # tempo de treino
    inicio_treino = time.time()
    modelo = treinar_naive_bayes_univariado(X_train, y_train)
    fim_treino = time.time()
    
    # tempo de teste
    inicio_teste = time.time()
    y_pred = [prever_univariado(modelo, x) for x in X_test]
    fim_teste = time.time()
    
    acc = accuracy(y_test, y_pred)
    prec, rec, f1 = precision_recall_f1(y_test, y_pred)
    
    return acc, prec, f1, (fim_treino - inicio_treino), (fim_teste - inicio_teste)

avaliacao multivariado

In [6]:
def avaliar_multivariado(X_train, y_train, X_test, y_test):
    modelo = treinar_bayes_multivariado(X_train, y_train)
    
    y_pred = []
    for x in X_test:
        y_pred.append(prever_multivariado(modelo, x))
    
    acc = accuracy(y_test, y_pred)
    prec, rec, f1 = precision_recall_f1(y_test, y_pred)
    
    return acc, prec, f1

In [31]:
def avaliar_multivariado(X_train, y_train, X_test, y_test):
    inicio_treino = time.time()
    modelo = treinar_bayes_multivariado(X_train, y_train)
    fim_treino = time.time()
    
    inicio_teste = time.time()
    y_pred = [prever_multivariado(modelo, x) for x in X_test]
    fim_teste = time.time()
    
    acc = accuracy(y_test, y_pred)
    prec, rec, f1 = precision_recall_f1(y_test, y_pred)
    
    return acc, prec, f1, (fim_treino - inicio_treino), (fim_teste - inicio_teste)

kfol manual

In [7]:
import random

def k_fold_split(X, y, k=5):
    dados = list(zip(X, y))
    random.shuffle(dados)
    
    fold_size = len(dados) // k
    folds = []
    
    for i in range(k):
        inicio = i * fold_size
        fim = inicio + fold_size
        folds.append(dados[inicio:fim])
    
    return folds

In [8]:
def avaliar_kfold(X, y, k=5, tipo="uni"):
    folds = k_fold_split(X, y, k)
    
    accs, precs, f1s = [], [], []
    
    for i in range(k):
        teste = folds[i]
        treino = []
        
        for j in range(k):
            if j != i:
                treino.extend(folds[j])
        
        X_train = [x for x, y in treino]
        y_train = [y for x, y in treino]
        
        X_test = [x for x, y in teste]
        y_test = [y for x, y in teste]
        
        if tipo == "uni":
            acc, prec, f1 = avaliar_univariado(X_train, y_train, X_test, y_test)
        else:
            acc, prec, f1 = avaliar_multivariado(X_train, y_train, X_test, y_test)
        
        accs.append(acc)
        precs.append(prec)
        f1s.append(f1)
    
    return accs, precs, f1s

In [32]:
def avaliar_kfold(X, y, k=5, tipo="uni"):
    folds = k_fold_split(X, y, k)
    
    accs, precs, f1s = [], [], []
    tempos_treino, tempos_teste = [], []
    
    for i in range(k):
        teste = folds[i]
        treino = []
        
        for j in range(k):
            if j != i:
                treino.extend(folds[j])
        
        X_train = [x for x, y in treino]
        y_train = [y for x, y in treino]
        
        X_test = [x for x, y in teste]
        y_test = [y for x, y in teste]
        
        if tipo == "uni":
            acc, prec, f1, t_train, t_test = avaliar_univariado(X_train, y_train, X_test, y_test)
        else:
            acc, prec, f1, t_train, t_test = avaliar_multivariado(X_train, y_train, X_test, y_test)
        
        accs.append(acc)
        precs.append(prec)
        f1s.append(f1)
        tempos_treino.append(t_train)
        tempos_teste.append(t_test)
    
    return accs, precs, f1s, tempos_treino, tempos_teste

In [33]:
import numpy as np

def formatar(valores):
    return f"{np.mean(valores):.2f} ± {np.std(valores):.2f}"

In [9]:
import numpy as np

def resumo_metricas(valores):
    return np.mean(valores), np.std(valores)

meus dados

In [10]:
def carregar_arff(caminho):
    X = []
    y = []
    lendo_dados = False

    with open(caminho, 'r') as f:
        for linha in f:
            linha = linha.strip()

            if linha == "" or linha.startswith("%"):
                continue

            if linha.lower() == "@data":
                lendo_dados = True
                continue

            if lendo_dados:
                partes = linha.split(",")

                # última coluna = classe
                atributos = partes[:-1]
                classe = partes[-1]

                # converter números quando possível
                nova_linha = []
                for val in atributos:
                    try:
                        nova_linha.append(float(val))
                    except:
                        nova_linha.append(val)

                X.append(nova_linha)
                y.append(classe)

    return X, y

In [11]:
Xc, yc = carregar_arff("datasets/BNG(credit-g)_classificacao.arff")
Xr, yr = carregar_arff("datasets/file22f1627e4a960_regressao.arff")

In [12]:
def codificar_categoricos(X):
    mapas = [{} for _ in range(len(X[0]))]

    for j in range(len(X[0])):
        valor_id = 0
        for i in range(len(X)):
            val = X[i][j]

            if isinstance(val, str):
                if val not in mapas[j]:
                    mapas[j][val] = valor_id
                    valor_id += 1
                X[i][j] = mapas[j][val]

    return X

In [13]:
Xc_numerico = codificar_categoricos(Xc)
Xr_numerico = codificar_categoricos(Xr)

In [14]:
def amostra_estratificada(X, y, n_total):
    from collections import defaultdict
    import random

    grupos = defaultdict(list)

    for i in range(len(y)):
        grupos[y[i]].append(i)

    X_out, y_out = [], []

    for classe, indices in grupos.items():
        n_classe = int(len(indices) / len(y) * n_total)
        escolhidos = random.sample(indices, n_classe)

        for i in escolhidos:
            X_out.append(X[i])
            y_out.append(y[i])

    return X_out, y_out

In [15]:
Xc_numerico_novo, yc_novo = amostra_estratificada(Xc_numerico, yc, 10000)

In [16]:
def normalizar(X):
    mins = [min(col) for col in zip(*X)]
    maxs = [max(col) for col in zip(*X)]

    X_norm = []
    for linha in X:
        nova = []
        for i in range(len(linha)):
            if maxs[i] == mins[i]:
                nova.append(0)
            else:
                nova.append((linha[i] - mins[i]) / (maxs[i] - mins[i]))
        X_norm.append(nova)

    return X_norm

In [17]:
Xc_normalizado = normalizar(Xc_numerico_novo)
Xr_normalizado = normalizar(Xr_numerico)

In [18]:
X1 = Xc_normalizado
y1 = yc_novo

X2 = Xr_normalizado
y2 = yr

avaliacao final

In [19]:
# classificação
accs, precs, f1s = avaliar_kfold(X1, y1, k=3, tipo="uni")

print("Naive Bayes Univariado:")
print("Accuracy:", resumo_metricas(accs))
print("Precision:", resumo_metricas(precs))
print("F1:", resumo_metricas(f1s))


Naive Bayes Univariado:
Accuracy: (np.float64(0.61996199619962), np.float64(0.014124443280519))
Precision: (np.float64(0.6283859655955426), np.float64(0.006469651783463383))
F1: (np.float64(0.606102215094568), np.float64(0.006014129115305559))


In [20]:
accs, precs, f1s = avaliar_kfold(X1, y1, k=3, tipo="multi")

print("\nNaive Bayes Multivariado:")
print("Accuracy:", resumo_metricas(accs))
print("Precision:", resumo_metricas(precs))
print("F1:", resumo_metricas(f1s))


Naive Bayes Multivariado:
Accuracy: (np.float64(0.6451645164516452), np.float64(0.015228472640311866))
Precision: (np.float64(0.6479046689910969), np.float64(0.0005633001018973992))
F1: (np.float64(0.6307547754242298), np.float64(0.009200550474416863))


In [ ]:
len(X2) #9999
len(X2) #10692  

10692

In [26]:
# classificação
accs, precs, f1s = avaliar_kfold(X2, y2, k=3, tipo="uni")

print("Naive Bayes Univariado:")
print("Accuracy:", resumo_metricas(accs))
print("Precision:", resumo_metricas(precs))
print("F1:", resumo_metricas(f1s))

Naive Bayes Univariado:
Accuracy: (np.float64(0.02600074822297045), np.float64(0.0024064123326353556))
Precision: (np.float64(0.011120458394338804), np.float64(0.0007899107387664185))
F1: (np.float64(0.010920973799663431), np.float64(0.0003928445425066719))


In [29]:
accs, precs, f1s = avaliar_kfold(X2, y2, k=3, tipo="multi")

print("\nNaive Bayes Multivariado:")
print("Accuracy:", resumo_metricas(accs))
print("Precision:", resumo_metricas(precs))
print("F1:", resumo_metricas(f1s))

C:\Users\ADM\AppData\Local\Temp\ipykernel_5796\2768425741.py:9: RuntimeWarning: Degrees of freedom <= 0 for slice
  sigma = np.cov(dados, rowvar=False)
C:\Users\ADM\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\ADM\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\ADM\AppData\Roaming\Python\Python313\site-packages\numpy\linalg\_linalg.py:2431: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)



Naive Bayes Multivariado:
Accuracy: (np.float64(0.031612420501309385), np.float64(0.0011301015687985943))
Precision: (np.float64(0.014654372624397607), np.float64(0.0010176379587369458))
F1: (np.float64(0.01448480696637758), np.float64(0.0005290626667343195))


tabela

In [37]:
# UNIVARIADO
accs, precs, f1s, t_train, t_test = avaliar_kfold(X1, y1, k=3, tipo="uni")

uni_acc = formatar(accs)
uni_prec = formatar(precs)
uni_f1 = formatar(f1s)
uni_train = formatar(t_train)
uni_test = formatar(t_test)



In [39]:
print(uni_acc, uni_prec, uni_f1, uni_train, uni_test)

0.62 ± 0.01 0.63 ± 0.00 0.61 ± 0.01 0.01 ± 0.00 0.10 ± 0.00


In [ ]:
# MULTIVARIADO
accs, precs, f1s, t_train, t_test = avaliar_kfold(X2, y2, k=3, tipo="multi")

multi_acc = formatar(accs)
multi_prec = formatar(precs)
multi_f1 = formatar(f1s)
multi_train = formatar(t_train)
multi_test = formatar(t_test)

C:\Users\ADM\AppData\Local\Temp\ipykernel_5796\2768425741.py:9: RuntimeWarning: Degrees of freedom <= 0 for slice
  sigma = np.cov(dados, rowvar=False)
C:\Users\ADM\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\ADM\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\ADM\AppData\Roaming\Python\Python313\site-packages\numpy\linalg\_linalg.py:2431: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


In [ ]:
print(multi_acc, multi_prec, multi_f1, multi_train, multi_test)